# Step 1: Launch SageMaker Processing for Multimodal Data Prep

Uses `ScriptProcessor` with the sklearn container to run `preprocess.py`.

**Input:** Raw JSONL files in S3.  
**Output:** Filtered train/validation/test JSONL splits in S3.

## Configuration

In [ ]:
import boto3
import sagemaker

REGION = boto3.session.Session().region_name
sess = sagemaker.session.Session()
BUCKET = sess.default_bucket()
S3_PREFIX = "autogluon-multimodal"

S3_INPUT = f"s3://{BUCKET}/{S3_PREFIX}/raw/"
S3_OUTPUT = f"s3://{BUCKET}/{S3_PREFIX}/processed"
INSTANCE_TYPE = "ml.m5.xlarge"

print(f"Region:      {REGION}")
print(f"S3 input:    {S3_INPUT}")
print(f"S3 output:   {S3_OUTPUT}")

## Discover IAM Role

In [ ]:
iam = boto3.client("iam")
role_arn = None
paginator = iam.get_paginator("list_roles")
for page in paginator.paginate():
    for role in page["Roles"]:
        if "SageMaker" in role["RoleName"] or "sagemaker" in role["RoleName"]:
            role_arn = role["Arn"]
            break
    if role_arn:
        break

if not role_arn:
    raise ValueError("No SageMaker IAM role found. Set role_arn manually.")

print(f"Role: {role_arn}")

## Imports and Processing Image

In [ ]:
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session
from sagemaker.core.processing import (
    ProcessingInput,
    ProcessingOutput,
    ScriptProcessor,
)
from sagemaker.core.shapes.shapes import ProcessingS3Input, ProcessingS3Output

session = Session()

processing_image = image_uris.retrieve("sklearn", region=REGION, version="1.2-1")
print(f"Processing image: {processing_image}")

## Create ScriptProcessor

In [ ]:
processor = ScriptProcessor(
    image_uri=processing_image,
    role=role_arn,
    command=["python3"],
    instance_type=INSTANCE_TYPE,
    instance_count=1,
    sagemaker_session=session,
)

## Run Processing Job

This launches a SageMaker Processing job that runs `preprocess.py` inside the sklearn container.

In [ ]:
processor.run(
    code="preprocess.py",
    inputs=[
        ProcessingInput(
            input_name="input",
            s3_input=ProcessingS3Input(
                s3_uri=S3_INPUT,
                local_path="/opt/ml/processing/input",
                s3_data_type="S3Prefix",
            ),
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            s3_output=ProcessingS3Output(
                s3_uri=f"{S3_OUTPUT}/train/",
                local_path="/opt/ml/processing/train",
                s3_upload_mode="EndOfJob",
            ),
        ),
        ProcessingOutput(
            output_name="validation",
            s3_output=ProcessingS3Output(
                s3_uri=f"{S3_OUTPUT}/validation/",
                local_path="/opt/ml/processing/validation",
                s3_upload_mode="EndOfJob",
            ),
        ),
        ProcessingOutput(
            output_name="test",
            s3_output=ProcessingS3Output(
                s3_uri=f"{S3_OUTPUT}/test/",
                local_path="/opt/ml/processing/test",
                s3_upload_mode="EndOfJob",
            ),
        ),
    ],
    wait=True,
    logs=True,
)

print(f"\nProcessing complete.")
print(f"Train data:      {S3_OUTPUT}/train/")
print(f"Validation data: {S3_OUTPUT}/validation/")
print(f"Test data:       {S3_OUTPUT}/test/")

## Next Steps

Run `1-training/launch_training.ipynb` with these S3 paths.